In [2]:
from torch.utils.data import DataLoader
import datetime as dt
from rl_stocks.utils.pipeline import prepare_training_data


In [3]:
sector_mix=['AAPL','MSFT','GOOGL','AMZN','TSLA','JNJ','PG','XOM','CVX','JPM','BAC','WMT','KO','NVDA','META','PFE','UNH','V','HD','NFLX','SBUX','NEE','MRK','GAP','AVGO','BA','T','GM','COST','ADI','DOW','GS','CAT','CRM','AMAT','NKE','GE','TWLO','PYPL']

start = dt.date.today() - dt.timedelta(days=365 * 5)
end = dt.date.today()


prepared = prepare_training_data(sector_mix, start, end, sequence_length=10)
full_dataset = prepared["dataset"]          # CVaRDataset

train_loader = DataLoader(full_dataset, batch_size=32, shuffle=True)

input_shape = full_dataset.state_shape      # pour DQNAgent / DDQNAgent
n_tickers = full_dataset.n_tickers


In [ ]:

# Train DDQNAgent:
ddqn_agent = DDQNAgent(input_shape, n_tickers)
ddqn_losses, ddqn_rewards = ddqn_agent.train(train_loader, epochs=20)

# Train DQNAgent:
print("processing for DQN agent.")
dqn_agent = DQNAgent(input_shape, n_tickers)
dqn_losses, dqn_rewards = dqn_agent.train(train_loader, epochs=20)  


# Plot the training losses for both agents:
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(ddqn_losses, label='DDQN Losses', color='blue')
plt.plot(dqn_losses, label='DQN Losses', color='orange')
plt.title("Training Loss Comparison")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()


if ddqn_rewards is not None and dqn_rewards is not None:
    plt.figure(figsize=(12, 5))  

    # Plot DDQNAgent rewards:
    plt.subplot(1, 2, 1)
    plt.plot(ddqn_rewards, label='DDQN Rewards', color='blue')
    plt.title("DDQNAgent Average Rewards")
    plt.xlabel("Epoch")
    plt.ylabel("Reward")
    plt.legend()

    # Plot DQNAgent rewards:
    plt.subplot(1, 2, 2)
    plt.plot(dqn_rewards, label='DQN Rewards', color='green')
    plt.title("DQNAgent Average Rewards")
    plt.xlabel("Epoch")
    plt.ylabel("Reward")
    plt.legend()

    plt.tight_layout()
    plt.show()

# Evaluate state and Calculate tickers weights:

state = full_dataset[0][0].clone().detach()  # Ensure proper tensor handling

# Compute weights from DDQNAgent:
ddqn_weights = ddqn_agent.get_weights(state)  
ddqn_weights_dict = dict(zip(Tickers, ddqn_weights))  # Match DDQN weights to tickers

# Compute weights from DQNAgent:
dqn_weights = dqn_agent.get_weights(state)  
dqn_weights_dict = dict(zip(Tickers, dqn_weights))  # Match DQN weights to tickers

# Combine tickers, DDQN weights, and DQN weights into a DataFrame:
weights_df = pd.DataFrame({
    'Ticker': Tickers,
    'DDQN_Weight': ddqn_weights,
    'DQN_Weight': dqn_weights
})

# Sort by DDQN weights in descending order: 
weights_df = weights_df.sort_values(by="DDQN_Weight", ascending=False).reset_index(drop=True)

print(weights_df)


# DQN, DDQN portfolios vs GSPC :

# Split data into train (75%) and test (25%):
train_size = int(0.75 * len(returns))
test_returns = returns.iloc[train_size:]
market_test_returns= market_returns.iloc[train_size:]

# Calculate portfolio cumulative returns:
ddqn_test_returns = test_returns.dot(ddqn_weights)
ddqn_cumulative_returns = (1 + ddqn_test_returns).cumprod() - 1
dqn_test_returns = test_returns.dot(dqn_weights)
dqn_cumulative_returns = (1 + dqn_test_returns).cumprod() - 1

# Calculate market cumulative returns ( ^GSPC):
gspc_test_returns=market_test_returns['^GSPC']
gspc_cumulative_returns = (1 + gspc_test_returns).cumprod() - 1

# Equal weights portfolio:
num_assets = len(Tickers)  
equal_weights = {ticker: 1/num_assets for ticker in Tickers}
  
equal_weights = pd.Series(equal_weights, index=test_returns.columns)

ew_test_returns = test_returns.dot(equal_weights)

ew_cumulative_returns = (1 + ew_test_returns).cumprod() - 1

# Create a DataFrame for comparison:
comparison_df = pd.DataFrame({
    'Date': test_returns.index,
    'DQN_Cumulative_Return': dqn_cumulative_returns,
    'DDQN_Cumulative_Return': ddqn_cumulative_returns,
    'Market_Cumulative_Return': gspc_cumulative_returns,
    'EW_Cumulative_Return': ew_cumulative_returns

})

# Plot cumulative returns:
plt.figure(figsize=(12, 6))
plt.plot(comparison_df['Date'], comparison_df['DQN_Cumulative_Return'], label='DQN Cumulative Return')
plt.plot(comparison_df['Date'], comparison_df['DDQN_Cumulative_Return'], label='DDQN Cumulative Return',color='red')
plt.plot(comparison_df['Date'], comparison_df['Market_Cumulative_Return'], label='Market (^GSPC) Cumulative Return')
plt.plot(comparison_df['Date'], comparison_df['EW_Cumulative_Return'], label='EW Cumulative Return')

plt.xlabel('Date Index')
plt.ylabel('Cumulative Return')
plt.title('DQN-DDQN vs. Broad Market Cumulative Returns on Test Data')
plt.legend()
plt.grid()
plt.show()


def calculate_performance(returns, threshold=0.0, confidence_level=0.95, risk_free_rate=0.04):
    cagr = (1 + returns).prod()**(252 / len(returns)) - 1
    volatility = returns.std() * np.sqrt(252)
    sharpe_ratio = cagr / volatility
    max_drawdown = (returns.cummax() - returns).max()

    # Omega Ratio
    positive_returns = returns[returns > threshold]
    negative_returns = returns[returns <= threshold]
    omega_ratio = positive_returns.sum() / abs(negative_returns.sum())

    # CVaR
    var = returns.quantile(1 - confidence_level)
    cvar = returns[returns <= var].mean()

    # Sortino Ratio
    downside_risk = returns[returns < risk_free_rate].std() * np.sqrt(252)
    sortino_ratio = (cagr - risk_free_rate) / downside_risk

    return cagr, volatility, sharpe_ratio, max_drawdown, omega_ratio, cvar, sortino_ratio

ddqn_metrics = calculate_performance(ddqn_test_returns)
dqn_metrics = calculate_performance(dqn_test_returns)
gspc_metrics=calculate_performance(gspc_test_returns)
ew_metrics=calculate_performance(ew_test_returns)

def create_metrics_dataframe(dq_metrics, ddq_metrics, benchmark_metrics, ew_metrics):
    """
    Creates a DataFrame with performance metrics for Portfolio, Benchmark, and Equal Weights.

    Args:
    - portfolio_metrics (list or tuple): Metrics for the portfolio.
    - benchmark_metrics (list, tuple, or pandas Series): Metrics for the benchmark.
    - ew_metrics (list or tuple): Metrics for the equal weights portfolio.

    Returns:
    - pd.DataFrame: A DataFrame with rows as metrics and columns as portfolios.
    """
    # Define row labels for the metrics:
    metrics_labels = ["CAGR", "Volatility", "Sharpe Ratio", "Max Drawdown", "Omega Ratio", "CVaR", "Sortino Ratio"]

    # Create the DataFrame using the compact dictionary as input:
    data = {
        "DQN": [float(value) for value in dqn_metrics],
        "DDQN": [float(value) for value in ddqn_metrics],
        "Benchmark": [float(value.iloc[0]) if hasattr(value, 'iloc') else float(value) for value in gspc_metrics],
        "EW": [float(value) for value in ew_metrics],
        
    }

    # Create the DataFrame:
    performance_df = pd.DataFrame(data, index=metrics_labels)

    return performance_df

performance_df = create_metrics_dataframe(dqn_metrics, ddqn_metrics,gspc_metrics, ew_metrics)
print(performance_df)

In [ ]:

# Preview the resulting DataFrame:
print("Shape of merged_data:", merged_data.shape)
print(merged_data.head())

columns_order = ['Date'] + [col for col in merged_data.columns if col != 'Date']
merged_data = merged_data[columns_order]


# Identify feature columns: 
feature_columns = [col for col in merged_data.columns if col not in ['Date', 'Ticker', 'Action']]

# Initialize the scaler:
scaler = StandardScaler()

# Fit and transform the feature columns:
merged_data[feature_columns] = scaler.fit_transform(merged_data[feature_columns])

# Split `merged_data` into `feature_data` and `action_labels`:

feature_df = merged_data[feature_columns]
action_labels_df = merged_data[['Ticker','Action']]

# Reindex action_labels_df to match features (trimmed_feature_data):
action_labels_df.index = trimmed_feature_data.index
grouped_cvar_df=grouped_cvar_df.drop(columns='Date')
grouped_cvar_df = grouped_cvar_df.iloc[:len(action_labels_df)]

# Create a dictionary mapping tickers to their CVaR values (first occurrence):
cvar_dict = grouped_cvar_df.groupby('Ticker')['CVaR'].first().to_dict()

# Map the CVaR values to `trimmed_feature_data` using the Ticker column:
aligned_cvar_df = trimmed_feature_data[['Ticker']].copy()
aligned_cvar_df['CVaR'] = aligned_cvar_df['Ticker'].map(cvar_dict)

#  Verify the result (shape and contents):
assert aligned_cvar_df.shape == (trimmed_feature_data.shape[0], 2)

aligned_cvar_df.head()

# Initialize the Dataset:

sequence_length = 10 
full_dataset = CVaRDataset(trimmed_feature_data, action_labels_df, aligned_cvar_df, sequence_length=sequence_length)

# Split the dataset into train and test indices:

train_indices, test_indices = train_test_split(
    list(range(len(full_dataset))),
    test_size=0.1,
    random_state=42
)

# Split into train and test subsets:
train_dataset = torch.utils.data.Subset(full_dataset, train_indices)
test_dataset = torch.utils.data.Subset(full_dataset, test_indices)
print("Number of training samples:", len(train_indices))
print("Number of testing samples:", len(test_indices))

# Create data loaders:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)

input_shape = full_dataset[0][0].shape  # Get the shape of the input state
n_tickers = len(Tickers)

# Train DDQNAgent:
ddqn_agent = DDQNAgent(input_shape, n_tickers)
ddqn_losses, ddqn_rewards = ddqn_agent.train(train_loader, epochs=20)

# Train DQNAgent:
print("processing for DQN agent.")
dqn_agent = DQNAgent(input_shape, n_tickers)
dqn_losses, dqn_rewards = dqn_agent.train(train_loader, epochs=20)  


# Plot the training losses for both agents:
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(ddqn_losses, label='DDQN Losses', color='blue')
plt.plot(dqn_losses, label='DQN Losses', color='orange')
plt.title("Training Loss Comparison")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()


if ddqn_rewards is not None and dqn_rewards is not None:
    plt.figure(figsize=(12, 5))  

    # Plot DDQNAgent rewards:
    plt.subplot(1, 2, 1)
    plt.plot(ddqn_rewards, label='DDQN Rewards', color='blue')
    plt.title("DDQNAgent Average Rewards")
    plt.xlabel("Epoch")
    plt.ylabel("Reward")
    plt.legend()

    # Plot DQNAgent rewards:
    plt.subplot(1, 2, 2)
    plt.plot(dqn_rewards, label='DQN Rewards', color='green')
    plt.title("DQNAgent Average Rewards")
    plt.xlabel("Epoch")
    plt.ylabel("Reward")
    plt.legend()

    plt.tight_layout()
    plt.show()

# Evaluate state and Calculate tickers weights:

state = full_dataset[0][0].clone().detach()  # Ensure proper tensor handling

# Compute weights from DDQNAgent:
ddqn_weights = ddqn_agent.get_weights(state)  
ddqn_weights_dict = dict(zip(Tickers, ddqn_weights))  # Match DDQN weights to tickers

# Compute weights from DQNAgent:
dqn_weights = dqn_agent.get_weights(state)  
dqn_weights_dict = dict(zip(Tickers, dqn_weights))  # Match DQN weights to tickers

# Combine tickers, DDQN weights, and DQN weights into a DataFrame:
weights_df = pd.DataFrame({
    'Ticker': Tickers,
    'DDQN_Weight': ddqn_weights,
    'DQN_Weight': dqn_weights
})

# Sort by DDQN weights in descending order: 
weights_df = weights_df.sort_values(by="DDQN_Weight", ascending=False).reset_index(drop=True)

print(weights_df)


# DQN, DDQN portfolios vs GSPC :

# Split data into train (75%) and test (25%):
train_size = int(0.75 * len(returns))
test_returns = returns.iloc[train_size:]
market_test_returns= market_returns.iloc[train_size:]

# Calculate portfolio cumulative returns:
ddqn_test_returns = test_returns.dot(ddqn_weights)
ddqn_cumulative_returns = (1 + ddqn_test_returns).cumprod() - 1
dqn_test_returns = test_returns.dot(dqn_weights)
dqn_cumulative_returns = (1 + dqn_test_returns).cumprod() - 1

# Calculate market cumulative returns ( ^GSPC):
gspc_test_returns=market_test_returns['^GSPC']
gspc_cumulative_returns = (1 + gspc_test_returns).cumprod() - 1

# Equal weights portfolio:
num_assets = len(Tickers)  
equal_weights = {ticker: 1/num_assets for ticker in Tickers}
  
equal_weights = pd.Series(equal_weights, index=test_returns.columns)

ew_test_returns = test_returns.dot(equal_weights)

ew_cumulative_returns = (1 + ew_test_returns).cumprod() - 1

# Create a DataFrame for comparison:
comparison_df = pd.DataFrame({
    'Date': test_returns.index,
    'DQN_Cumulative_Return': dqn_cumulative_returns,
    'DDQN_Cumulative_Return': ddqn_cumulative_returns,
    'Market_Cumulative_Return': gspc_cumulative_returns,
    'EW_Cumulative_Return': ew_cumulative_returns

})

# Plot cumulative returns:
plt.figure(figsize=(12, 6))
plt.plot(comparison_df['Date'], comparison_df['DQN_Cumulative_Return'], label='DQN Cumulative Return')
plt.plot(comparison_df['Date'], comparison_df['DDQN_Cumulative_Return'], label='DDQN Cumulative Return',color='red')
plt.plot(comparison_df['Date'], comparison_df['Market_Cumulative_Return'], label='Market (^GSPC) Cumulative Return')
plt.plot(comparison_df['Date'], comparison_df['EW_Cumulative_Return'], label='EW Cumulative Return')

plt.xlabel('Date Index')
plt.ylabel('Cumulative Return')
plt.title('DQN-DDQN vs. Broad Market Cumulative Returns on Test Data')
plt.legend()
plt.grid()
plt.show()


def calculate_performance(returns, threshold=0.0, confidence_level=0.95, risk_free_rate=0.04):
    cagr = (1 + returns).prod()**(252 / len(returns)) - 1
    volatility = returns.std() * np.sqrt(252)
    sharpe_ratio = cagr / volatility
    max_drawdown = (returns.cummax() - returns).max()

    # Omega Ratio
    positive_returns = returns[returns > threshold]
    negative_returns = returns[returns <= threshold]
    omega_ratio = positive_returns.sum() / abs(negative_returns.sum())

    # CVaR
    var = returns.quantile(1 - confidence_level)
    cvar = returns[returns <= var].mean()

    # Sortino Ratio
    downside_risk = returns[returns < risk_free_rate].std() * np.sqrt(252)
    sortino_ratio = (cagr - risk_free_rate) / downside_risk

    return cagr, volatility, sharpe_ratio, max_drawdown, omega_ratio, cvar, sortino_ratio

ddqn_metrics = calculate_performance(ddqn_test_returns)
dqn_metrics = calculate_performance(dqn_test_returns)
gspc_metrics=calculate_performance(gspc_test_returns)
ew_metrics=calculate_performance(ew_test_returns)

def create_metrics_dataframe(dq_metrics, ddq_metrics, benchmark_metrics, ew_metrics):
    """
    Creates a DataFrame with performance metrics for Portfolio, Benchmark, and Equal Weights.

    Args:
    - portfolio_metrics (list or tuple): Metrics for the portfolio.
    - benchmark_metrics (list, tuple, or pandas Series): Metrics for the benchmark.
    - ew_metrics (list or tuple): Metrics for the equal weights portfolio.

    Returns:
    - pd.DataFrame: A DataFrame with rows as metrics and columns as portfolios.
    """
    # Define row labels for the metrics:
    metrics_labels = ["CAGR", "Volatility", "Sharpe Ratio", "Max Drawdown", "Omega Ratio", "CVaR", "Sortino Ratio"]

    # Create the DataFrame using the compact dictionary as input:
    data = {
        "DQN": [float(value) for value in dqn_metrics],
        "DDQN": [float(value) for value in ddqn_metrics],
        "Benchmark": [float(value.iloc[0]) if hasattr(value, 'iloc') else float(value) for value in gspc_metrics],
        "EW": [float(value) for value in ew_metrics],
        
    }

    # Create the DataFrame:
    performance_df = pd.DataFrame(data, index=metrics_labels)

    return performance_df

performance_df = create_metrics_dataframe(dqn_metrics, ddqn_metrics,gspc_metrics, ew_metrics)
print(performance_df)